In [1]:
import hashlib

In [2]:
class User:
    def __init__(self,username,password) -> None:
        '''Create a new user object.The password will be encrypted before storing.'''
        self.username = username
        self.password = self._encrypt_pw(password)
        self.is_logged_in = False
    
    def _encrypt_pw(self,password):
        '''Encrypt the password with the username and return the sha digest.'''
        hash_string = (self.username + password)
        hash_string = hash_string.encode("utf8")
        return hashlib.sha256(hash_string).hexdigest()

    def check_password(self,password):
        '''Return True if the password is valid for this user,false otherwise.'''
        encrypted = self._encrypt_pw(password)
        return encrypted == self.password

In [3]:
class AuthException(Exception):
    def __init__(self, username,user=None) -> None:
        super().__init__(username,user)
        self.username = username
        self.user = user

class UsernameAlreadyExists(AuthException):
    pass

class PasswordTooSHort(AuthException):
    pass

In [4]:
class Authenticator:
    def __init__(self) -> None:
        '''Construct an authenticator to manage users logging in and out.'''
        self.users = {}

    def add_user(self,username,password):
        if username in self.users:
            raise UsernameAlreadyExists(username)
        if len(password) < 6:
            raise PasswordTooSHort(username)
        self.users[username] = User(username,password)
    
    def login(self,username,password):
        try:
            user = self.users[username]
        except KeyError:
            raise InvalidUsername(username)
        if not user.check_password(password):
            raise InvalidPassword(username,user)
        user.is_logged_in = True
        return True
    
    def is_logged_in(self,username):
        if username in self.users:
            return self.users[username].is_logged_in        
        return False


In [5]:
class InvalidUsername(AuthException):
    pass

class InvalidPassword(AuthException):
    pass

class PermissionError(Authenticator):
    pass

class NotLoggedInError(AuthException):
    pass

class NotPermittedError(AuthException):
    pass

In [6]:
class Authorizor:
    def __init__(self,authenticator) -> None:
        self.authenticator = authenticator
        self.permissions = {}

    def add_permission(self,perm_name):
        '''Create a new permission that users can be added to'''
        try:
            perm_set = self.permissions[perm_name]
        except KeyError:
            self.permissions[perm_name] = set()
        else:
            raise PermissionError("Permission Exists")
    
    def permit_user(self,perm_name,username):
        '''Grant the given permission to the user'''
        try:
            perm_set = self.permissions[perm_name]
        except KeyError:
            raise PermissionError("Permission does not exist")
        else:
            if username not in self.authenticator.users:
                raise InvalidUsername(username)
            perm_set.add(username)
    
    def check_permission(self,perm_name,username):
        if not self.authenticator.is_logged_in(username):
            raise NotLoggedInError(username)
        try:
            perm_set = self.permissions[perm_name]
        except KeyError:
            raise NotPermittedError(username)
        else:
            return True

In [9]:
authenticator = Authenticator()
authorizor = Authorizor(authenticator)

In [10]:
authenticator.add_user("joe","joepassword")
authorizor.add_permission("paint")
authorizor.check_permission("paint","joe")

NotLoggedInError: ('joe', None)

In [11]:
authenticator.is_logged_in("joe")

False

In [12]:
authenticator.login("joe","joepassword")

True

In [13]:
authorizor.check_permission("mix","joe")

NotPermittedError: ('joe', None)

In [14]:
authorizor.permit_user("mix","joe")

TypeError: __init__() takes 1 positional argument but 2 were given